# Train linear heads outside fse
Train scikit-learn classification and regression heads, then import them into fse.
Start with a synthetic dataset to check API use. Then train and evaluate on the published SST-5 train and test splits.
Use raw means (`normalize=False`) for filtering: fse can fold a linear head into each word row.
A head trained on normalized means must retain that normalization and the full input dimension.
Install `requirements.txt` from this folder. All generated files stay in a temporary directory.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

2026-11-01 macOS-27.0.1-arm64-arm-64bit-Mach-O arm64


In [2]:
# Deliberately tiny, hand-built word rows. These demonstrate APIs, not model quality.
words = ["cat", "dog", "pet", "kitten", "puppy", "car", "bus", "road", "train", "truck"]
vectors = np.array([
    [1, 0, .2], [1, .1, 0], [1, 0, 0], [1, 0, .3], [1, .2, 0],
    [0, 1, .2], [0, 1, .1], [0, 1, 0], [0, 1, .3], [0, 1, .4],
], dtype=np.float32)
encoder = fse.Encoder.from_arrays(vectors, words=words, normalize=False)
texts = ["cat pet", "dog puppy", "car road", "bus truck"]

In [3]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

examples = [f"{a} {b}" for group in (words[:5], words[5:]) for a in group for b in group]
labels = np.repeat(["pet", "vehicle"], 25)
train, test = train_test_split(np.arange(len(examples)), test_size=.3, random_state=7, stratify=labels)
x = encoder.encode(examples, workers=1)
head = LogisticRegression(random_state=7).fit(x[train], labels[train])
classifier = fse.Classifier.from_sklearn(encoder, head)
predicted = classifier.predict([examples[i] for i in test], workers=1)
np.testing.assert_array_equal(predicted, head.predict(x[test]))
np.testing.assert_allclose(classifier.predict_proba(examples, workers=1), head.predict_proba(x), atol=2e-6)
classifier.save(work / "filter.safetensors", standalone=True)
loaded = fse.Classifier.load(work / "filter.safetensors")
np.testing.assert_array_equal(loaded.predict(texts, workers=1), classifier.predict(texts, workers=1))
print("Synthetic held-out classification accuracy:", accuracy_score(labels[test], predicted))

Synthetic held-out classification accuracy: 1.0


In [4]:
# A continuous synthetic target with a separate held-out split.
target = (2 * x[:, 0] - x[:, 1] + .5 * x[:, 2]).astype(np.float64)
regression = Ridge(alpha=.01).fit(x[train], target[train])
regressor = fse.Regressor.from_sklearn(encoder, regression)
actual = regressor.predict([examples[i] for i in test], workers=1)
np.testing.assert_allclose(actual.reshape(-1), regression.predict(x[test]), atol=2e-6)
regressor.save(work / "regressor.safetensors", standalone=True)
print("Synthetic held-out regression MAE:", mean_absolute_error(target[test], actual.reshape(-1)))

Synthetic held-out regression MAE: 0.00347218910853068


## Sentiment heads on SST-5
Use [Stanford Sentiment Treebank](https://nlp.stanford.edu/sentiment/) sentence labels, distributed by [SetFit/sst5](https://huggingface.co/datasets/SetFit/sst5).
Keep its published train and test splits. This example uses fixed hyperparameters and does not tune on the test set.
Classification predicts five sentiment classes. Regression treats the labels 0 through 4 as equally spaced sentiment scores.
That spacing is an explicit modeling assumption. The regression error is in label units, not a calibrated review rating.
Use GloVe-300 raw means so both imported heads can fold. This notebook downloads data into its temporary directory.

In [5]:
import json
from huggingface_hub import hf_hub_download
from gensim.models import KeyedVectors
from fse.gensim import from_gensim

dataset_revision = "e51bdcd8cd3a30da231967c1a249ba59361279a3"
def reviews(split):
    path = hf_hub_download("SetFit/sst5", f"{split}.jsonl", repo_type="dataset", revision=dataset_revision, cache_dir=work / "hub")
    with open(path, encoding="utf-8") as stream:
        data = [json.loads(line) for line in stream]
    return [r["text"] for r in data], np.array([r["label"] for r in data])
train_texts, train_labels = reviews("train")
test_texts, test_labels = reviews("test")
revision = "4e5ad402664236b85be45a0d107b95259ba8a909"
paths = []
for filename in ("glove-wiki-gigaword-300.model", "glove-wiki-gigaword-300.model.vectors.npy"):
    cached = Path.home() / ".cache/huggingface/hub/models--fse--glove-wiki-gigaword-300/snapshots" / revision / filename
    paths.append(cached if cached.is_file() else Path(hf_hub_download("fse/glove-wiki-gigaword-300", filename, revision=revision, cache_dir=work / "hub")))
if paths[0].parent != paths[1].parent:
    paths = [Path(hf_hub_download("fse/glove-wiki-gigaword-300", filename, revision=revision, cache_dir=work / "hub")) for filename in ("glove-wiki-gigaword-300.model", "glove-wiki-gigaword-300.model.vectors.npy")]
real_encoder = fse.Encoder.from_word_vectors(from_gensim(KeyedVectors.load(str(paths[0]), mmap="r")), normalize=False)
train_rows = real_encoder.encode(train_texts, workers=1)
test_rows = real_encoder.encode(test_texts, workers=1)
classification = LogisticRegression(C=1, max_iter=1000, random_state=7).fit(train_rows, train_labels)
clf = fse.Classifier.from_sklearn(real_encoder, classification)
probabilities = clf.predict_proba(test_texts, workers=1)
np.testing.assert_allclose(probabilities, classification.predict_proba(test_rows), atol=3e-6, rtol=3e-6)
labels = clf.predict(test_texts, workers=1)
regression = Ridge(alpha=1).fit(train_rows, train_labels.astype(float))
reg = fse.Regressor.from_sklearn(real_encoder, regression)
predictions = reg.predict(test_texts, workers=1)
np.testing.assert_allclose(predictions, regression.predict(test_rows), atol=3e-5, rtol=3e-5)
clf.save(work / "sst5-classifier.safetensors", standalone=True)
reg.save(work / "sst5-regressor.safetensors", standalone=True)
print({"dataset": "SST-5", "revision": dataset_revision, "train": len(train_texts), "test": len(test_texts),
       "classification_accuracy": accuracy_score(test_labels, labels),
       "classification_label_disagreements_vs_sklearn": int(np.count_nonzero(labels != classification.predict(test_rows))),
       "regression_MAE_label_units": mean_absolute_error(test_labels, predictions),
       "regression_baseline_MAE": mean_absolute_error(test_labels, np.full(len(test_labels), train_labels.mean()))})

/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr153-recipes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dataset': 'SST-5', 'revision': 'e51bdcd8cd3a30da231967c1a249ba59361279a3', 'train': 8544, 'test': 2210, 'classification_accuracy': 0.4117647058823529, 'classification_label_disagreements_vs_sklearn': 0, 'regression_MAE_label_units': 0.8967320919036865, 'regression_baseline_MAE': 1.1410663141661157}


In [6]:
scratch.cleanup()